# Stage 3 -- cross-attention between ligand atoms and protein residues

**This notebook must be run in one sitting.** The two caches total ~21 GB and live on
local disk, which is wiped when the runtime dies. We are deliberately not paying Drive
quota for something a 7-minute job can rebuild.

```
protein residues, ESM-2 650M   3.81M x 1280 fp16  =  9.8 GB   (~4.5 min)
ligand atoms,     ChemBERTa    7.3M  x  768 fp16  = 11.2 GB   (~3 min)
```

**Runtime -> Change runtime type -> L4 GPU, and High-RAM if offered.** High-RAM lets the
ligand cache sit in host memory instead of being re-read from disk 162,000 times per
epoch, which is the likely bottleneck.

## What is being tested

PLAPT pools each side and then combines them, so no tensor in the model has a cell for
"ligand atom j touches residue i" -- which is what a binding contact is. This replaces
the pooling with attention:

```
PLAPT:  pooled_protein = (1/L) * sum_i  residue_i          uniform weights
here:   context_j      =  sum_i w_ij * V(residue_i)        weights chosen per atom
```

Mean pooling is the special case `w_ij = 1/L`, so this is the same averaging operation
with the weights computed instead of frozen.

## The arms

| arm | what it is |
|---|---|
| `full` | learned cross-attention, one query per ligand atom |
| `prot_zero` | protein zeroed -> ligand-only ceiling |
| `mol_zero` | ligand zeroed -> protein-only ceiling |
| `pooled_query` | one query from the mean-pooled ligand -- the cheap variant, no atom-level resolution. Optional. |

The comparison that matters is `full` against the published mean-pooling result of
**0.7350**, on the identical split and the identical 20,000 test pairs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.44,<6"
import torch, transformers, psutil
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda:", torch.cuda.is_available())
print(f"host RAM: {psutil.virtual_memory().total/1e9:.0f} GB"
      "   (>40 GB = High-RAM, use --mol-ram below)")

## 1. Upload

Six files this time -- **more than the previous notebook**, because the training script
needs the pair table and the target scaling:

- `colab_upload/02_embed.py`
- `scripts/06_cross_attention.py`
- `data/processed/proteins.parquet`
- `data/processed/ligands.parquet`
- `data/processed/pairs.parquet`   <- new
- `data/processed/meta.json`       <- new (holds y_mean / y_std / the split seed)

In [ ]:
from google.colab import files
up = files.upload()
for n in sorted(up): print(" uploaded:", n)

In [ ]:
import os, shutil
OUT = '/content/emb'
os.makedirs(OUT, exist_ok=True)
print("cache dir:", OUT)
print(f"free local disk: {shutil.disk_usage('/content').free/1e9:.0f} GB"
      "   (need ~21 GB for both caches)")

## 2. Protein per-residue pass (~4.5 min, 9.8 GB)

Watch for:

- `integrity: ... r = 1.000000` -- recomputes the mean pool from the stored fp16 ragged
  array and correlates it against the fp32 mean from the forward pass. One number
  validating the layout, the offsets, the special-token masking and the precision. The
  script raises below 0.999 rather than handing you a corrupt cache.
- `NOTE: ... truncated` -- expect **~7.1%** at 1024. The validation run showed 1.0% only
  because `--limit 512` takes the first rows rather than a random sample.

In [ ]:
!python 02_embed.py --data . --out "$OUT" --save-residues --exclude-special --skip-ligands --prot-model facebook/esm2_t33_650M_UR50D --prot-max 1024 --tag _esm650 --prot-budget 8192

## 3. Train

The first invocation also builds the ligand per-atom cache (~3 min, 11.2 GB). It is
written once and reused by every arm and seed in the run -- ChemBERTa is frozen, so
computing it live would repeat the same work once per epoch.

**Drop `--mol-ram` if the cell above reported less than ~40 GB of host RAM.**

### Watch the first epoch's wall time

I can't predict the step time reliably -- it depends on how the ligand reads behave --
so check epoch 1 and extrapolate before walking away. `--log-epochs` prints each one.
If an epoch takes more than ~3 minutes, stop and we'll tune the batching rather than
let it run for hours.

In [ ]:
!python 06_cross_attention.py --data . --emb "$OUT" --prot-tag _esm650 --arms full --seeds 1 --max-epochs 25 --patience 4 --batch-size 64 --mol-ram --log-epochs

## 4. Reading the result

```
reference, same split and same 20,000 test pairs, mean pooling + concat:
  ESM-2 650M   full 0.7350   ligand-only 0.5524   protein-only 0.5705
  zero-parameter protein lookup table  0.5784
```

- **`full` clearly above 0.7350** -- the contact hypothesis survives.
- **`full` at or below 0.7350** -- the rewiring did not help. Given the learning curve was
  still climbing at +0.05 R per doubling with no sign of saturation, the likely reason is
  that 8,584 proteins is not enough supervision to learn contact geometry without
  structural labels. That is a real result, not a failed experiment.

## 5. If `full` looks promising

Add the ablation arms and a second seed.

In [ ]:
!python 06_cross_attention.py --data . --emb "$OUT" --prot-tag _esm650 --arms full,prot_zero,mol_zero --seeds 2 --max-epochs 25 --patience 4 --batch-size 64 --mol-ram --tag _grid

## 6. Pull the results down

Only the CSVs matter -- a few KB. The 21 GB of caches stay here and die with the runtime.

In [ ]:
from google.colab import files
import glob, os
for f in sorted(glob.glob('results/*.csv')):
    print(f"{os.path.getsize(f)/1024:7.1f} KB  {f}")
    files.download(f)